import subprocess, sys

# ── Detect GPU before importing torch ─────────────────────────────────────
result = subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    cc = result.stdout.strip().split('\n')[0].strip()
    major = int(cc.split('.')[0])
    print(f'GPU compute capability: {cc}')
    if major < 7:
        print('P100 detected (sm_60) — installing PyTorch 2.0.1 (last version supporting P100)...')
        subprocess.run([sys.executable,'-m','pip','install',
                        'torch==2.0.1+cu117','torchvision==0.15.2+cu117',
                        '--index-url','https://download.pytorch.org/whl/cu117',
                        '--quiet','--upgrade'], check=True)
        print('Compatible PyTorch installed.')
    else:
        print(f'GPU sm_{cc} is compatible — keeping default PyTorch.')
else:
    print('No GPU detected or nvidia-smi unavailable.')

# ── Install dependencies ───────────────────────────────────────────────────
for p in ['pykeen>=1.10.0','optuna>=3.0.0','scipy>=1.10.0','scikit-learn>=1.3.0']:
    subprocess.run([sys.executable,'-m','pip','install',p,'--quiet'], check=True)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')


In [ ]:
import subprocess, sys
for p in ['pykeen>=1.10.0','optuna>=3.0.0','scipy>=1.10.0','scikit-learn>=1.3.0']:
    subprocess.run([sys.executable,'-m','pip','install',p,'--quiet'], check=True)
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import torch
MODEL_NAME     = 'TransE'
INDICATION_REL = 'indication'
EMBEDDING_DIM  = 64    # reduced from 128 for speed
NUM_EPOCHS     = 100   # reduced from 300 for speed
BATCH_SIZE     = 2048  # increased from 512 for GPU efficiency
PATIENCE       = 10    # reduced from 15
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED    = 42
SPLITS_TO_RUN  = [0]

print(f'Model        : {MODEL_NAME}')
print(f'Device       : {DEVICE}')
print(f'Embedding dim: {EMBEDDING_DIM}')
print(f'Epochs       : {NUM_EPOCHS}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'Splits to run: {SPLITS_TO_RUN}')

In [ ]:
from pathlib import Path
import pandas as pd

WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

# Dynamically locate splits directory
_candidates = list(Path('/kaggle/input').rglob('slice_0'))
if not _candidates:
    raise FileNotFoundError("Could not find slice_0 — please add the weightedkgblend-splits dataset")
SPLITS = _candidates[0].parent
print(f'Splits found at: {SPLITS}')
for sl in sorted(SPLITS.iterdir()):
    print(f'  {sl.name}: {[f.name for f in sl.iterdir()]}')

In [ ]:
import torch, numpy as np, pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline

TOP_K = 50  # number of top predicted diseases to save per query

def get_preds(model, factory, relation, device, top_k=TOP_K):
    model.eval()
    rel_id = factory.relation_to_id.get(relation)
    if rel_id is None: return pd.DataFrame()
    triples = factory.mapped_triples
    mask = triples[:,1] == rel_id
    rows = []
    with torch.no_grad():
        for triple in triples[mask]:
            h,r,t = triple[0].item(), triple[1].item(), triple[2].item()
            hr = torch.tensor([[h,r]], device=device)
            scores = model.score_t(hr).squeeze(0)
            order  = torch.argsort(scores, descending=True).cpu().numpy()
            rank   = int(np.where(order==t)[0][0]) + 1
            row = {'drug': factory.entity_id_to_label[h],
                   'expected_disease': factory.entity_id_to_label[t],
                   'rank': rank, 'reciprocal_rank': 1.0/rank}
            for k in range(1, top_k + 1):
                row[f'top{k}_disease'] = factory.entity_id_to_label[order[k-1]] if k-1 < len(order) else ''
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
import time

PRED_DIR = WORK / 'predictions' / 'TransE'
PRED_DIR.mkdir(parents=True, exist_ok=True)

def print_metrics(df, split_name):
    mrr    = df.reciprocal_rank.mean()
    hits1  = (df['rank'] <= 1).mean()
    hits3  = (df['rank'] <= 3).mean()
    hits5  = (df['rank'] <= 5).mean()
    hits10 = (df['rank'] <= 10).mean()
    print(f'  {split_name} MRR={mrr:.4f}  Hits@1={hits1:.4f}  Hits@3={hits3:.4f}  Hits@5={hits5:.4f}  Hits@10={hits10:.4f}')

total_start = time.time()

for i in SPLITS_TO_RUN:
    sl     = SPLITS / f'slice_{i}'
    outdir = PRED_DIR / f'slice_{i}'
    outdir.mkdir(parents=True, exist_ok=True)

    if (outdir / 'predictions_test.tsv').exists():
        print(f'SKIP TransE/slice_{i} (already done)')
        continue

    print(f'\n==================================================')
    print(f'Training TransE — slice_{i}')
    print(f'==================================================')
    t0 = time.time()

    tf_train = TriplesFactory.from_labeled_triples(
        pd.read_csv(sl/'kge_train.tsv', sep='\t', header=None, names=['h','r','t']).values.astype(str))
    tf_test  = TriplesFactory.from_labeled_triples(
        pd.read_csv(sl/'ind_test.tsv',  sep='\t', header=None, names=['h','r','t']).values.astype(str),
        entity_to_id=tf_train.entity_to_id, relation_to_id=tf_train.relation_to_id)
    tf_valid = TriplesFactory.from_labeled_triples(
        pd.read_csv(sl/'ind_valid.tsv', sep='\t', header=None, names=['h','r','t']).values.astype(str),
        entity_to_id=tf_train.entity_to_id, relation_to_id=tf_train.relation_to_id)

    print(f'  Train triples : {len(tf_train.mapped_triples):,}')
    print(f'  Entities      : {tf_train.num_entities:,}')

    res = pipeline(
        training=tf_train, testing=tf_test, validation=tf_valid,
        model='TransE',
        model_kwargs=dict(embedding_dim=EMBEDDING_DIM),
        optimizer='Adam', optimizer_kwargs=dict(lr=0.001),
        training_kwargs=dict(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE),
        stopper='early', stopper_kwargs=dict(patience=PATIENCE),
        device=DEVICE, random_seed=RANDOM_SEED)

    for split_name, tf in [('test', tf_test), ('valid', tf_valid)]:
        preds = get_preds(res.model, tf, INDICATION_REL, DEVICE)
        preds.to_csv(outdir / f'predictions_{split_name}.tsv', sep='\t', index=False)
        print_metrics(preds, split_name)

    elapsed = time.time() - t0
    print(f'\n  slice_{i} done in {elapsed/60:.1f} min')

total_elapsed = time.time() - total_start
print(f'\nTotal elapsed: {total_elapsed/60:.1f} min')
print(f'Estimated time for all 5 splits: {total_elapsed/60 * 5 / len(SPLITS_TO_RUN):.1f} min')